# BMIN 5200 — Week 1 in-class exercise
## ELIZA: pattern matching and the appearance of understanding

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LINK::github-repo/blob/main/exercises/week01.ipynb)

**Time:** ~25 minutes · **Pairs with:** Course intro; history of AI (Turing Test, ELIZA, hype vs. realities)

### Tasks
- Confirm your Colab environment runs the code we will use all semester
- Implement ELIZA's core trick — an ordered list of regex patterns plus pronoun reflection — in about 20 lines
- Run a scripted patient-intake conversation against it, then deliberately break it
- Write down, on day one, what you think "understanding a patient complaint" would actually require

### Background
Weizenbaum wrote ELIZA in 1966 as a parody of Rogerian therapy, and was disturbed to find
that people confided in it and insisted it understood them. Its direct descendants are being
sold today as clinical intake chatbots, symptom checkers, and triage front doors — systems
that sit between a patient and a clinician. Knowing exactly how thin the machinery underneath
can be is the difference between evaluating one of these tools and being sold one.

## Setup

Run this cell first. It only uses the Python standard library plus `numpy`, both of which
Colab already has, so nothing installs. If it prints a version and a confirmation line, your
environment is working and you are set for the semester.

In [ ]:
import re
import sys

import numpy as np

# Fixed seed so every laptop in the room prints the same conversation and Joe can
# refer to a specific line out loud.
rng = np.random.default_rng(5200)

print("Python:", sys.version.split()[0])
print("numpy :", np.__version__)
print("Environment check: ready.")

## Part 1 — Pronoun reflection

ELIZA's first move is grammatical, not semantic. It takes the fragment of your sentence it
captured and swaps the pronouns, so *"my chest pain"* comes back as *"your chest pain"*. That
one substitution is what makes the output sound like it is addressed to you. Read the
dictionary below, then run the cell — there is no exercise here yet, just look at what
`reflect` does and does not do.

In [ ]:
REFLECTIONS = {
    "i": "you", "me": "you", "my": "your", "mine": "yours", "myself": "yourself",
    "i'm": "you are", "i've": "you have", "am": "are",
    "you": "I", "your": "my", "yours": "mine", "yourself": "myself",
    "are": "am", "was": "were",
}


def reflect(fragment):
    # Word-by-word substitution. Nothing here knows what a word means; a word is
    # either a key in the dictionary or it is passed through untouched.
    return " ".join(REFLECTIONS.get(word, word) for word in fragment.split())


for fragment in ["my chest pain", "i am short of breath", "my father", "my qwerty"]:
    print(f"{fragment:25s} -> {reflect(fragment)}")

## Part 2 — An ordered list of patterns

Everything else in ELIZA is a list of `(regex, responses)` pairs, tried **in order**, first
match wins. The captured group gets reflected and dropped into a response template. The last
rule matches anything, which is why ELIZA never has nothing to say.

The starter list below is deliberately incomplete. Run it, then look at the two `# TODO`
lines: a patient-intake bot that cannot recognize a medication or a family history is missing
the two things intake exists to collect.

In [ ]:
RULES = [
    # Order matters: the most specific patterns must come first, and the catch-all last.
    (r".*\bi (?:am|'m) (?:really |very )?(worried|scared|anxious) about (.*)",
     ["Why does {1} make you {0}?", "How long have you been {0} about {1}?"]),
    (r".*\bi (?:feel|am feeling) (.*)",
     ["Do you often feel {0}?", "Tell me more about feeling {0}."]),
    (r".*\bmy (.*) hurts?",
     ["When did your {0} start hurting?", "Tell me more about your {0}."]),
    (r".*\b(?:i|we) (?:can't|cannot|can not) (.*)",
     ["What stops you from being able to {0}?"]),

    # TODO: add a rule that captures a medication, e.g. matching "I take metoprolol
    #       every morning" and answering "How long have you been taking {0}?"

    # TODO: add a rule that reacts to any of mother / father / mom / dad / family /
    #       sister / brother with "Tell me more about your family."
    #       This rule captures nothing, so its template needs no {0}.

    (r"(.*)", ["Please, go on.", "Can you say more about that?", "I see. And what else?"]),
]


def respond(utterance, generator=rng):
    cleaned = re.sub(r"[.!?]+$", "", utterance.strip().lower())
    for pattern, templates in RULES:
        match = re.match(pattern, cleaned)
        if match:
            # Real ELIZA rotates among several phrasings so the repetition is less obvious.
            template = templates[int(generator.integers(len(templates)))]
            return template.format(*[reflect(group) for group in match.groups()])
    return "Please, go on."


print(len(RULES), "rules loaded")
print(respond("I am worried about my chest pain", np.random.default_rng(5200)))

### Predict before you run

The next cell moves the catch-all rule `(.*)` from the end of the list to the **front** and
re-runs four utterances that currently match specific rules. Commit out loud before running:
how many of the four still get a specific reply?

The answer is the whole reason ELIZA's rule list is ordered rather than a dictionary.

In [ ]:
RULES_BACKUP = RULES
RULES = [RULES_BACKUP[-1]] + RULES_BACKUP[:-1]   # catch-all first

probe = np.random.default_rng(5200)
for line in ["I am worried about my chest pain", "I feel dizzy",
             "My chest hurts", "I can't climb one flight of stairs"]:
    print(f"{line:38s} -> {respond(line, probe)}")

RULES = RULES_BACKUP   # put it back before we continue

All four, and with the same three words. A regex that matches everything, placed first,
makes every rule after it unreachable — the list is a sequence of tests, not a lookup table.
This is the earliest form of a *conflict resolution strategy*: when several rules could fire,
something has to decide which one does. ELIZA's strategy is "whichever I wrote first." In
Week 6 we will meet expert systems that make that decision explicitly, and in Week 9 we will
see what happens when the ordering is learned from data rather than written down.

## Part 3 — A scripted intake

Here is a seven-line patient transcript, of the kind an intake bot would see. This is
synthetic — no real patient said any of it. Before you run the cell: with the starter rules
(plus whatever you added in Part 2), **which lines will fall through to the catch-all?**
Count them out loud, then run.

In [ ]:
TRANSCRIPT = [
    "I am worried about my chest pain",
    "It hurts when I climb the stairs",
    "My shoulder hurts too",
    "I take metoprolol every morning",
    "I can't sleep lying flat",
    "My father died of a heart attack at 52",
    "I feel like something is really wrong",
]

conversation = np.random.default_rng(5200)
for line in TRANSCRIPT:
    print("PATIENT:", line)
    print("ELIZA  :", respond(line, conversation))
    print()

Line 2, *"It hurts when I climb the stairs"*, is worth pausing on. It falls through even
though it is the single most diagnostically loaded sentence in the transcript — exertional
chest pain is the finding that moves this patient toward a cardiac workup. ELIZA misses it
for a purely syntactic reason: the pattern wants the word `my` before the thing that hurts.
Nothing in the program is capable of noticing that this line matters more than the others.

## Part 4 — Failure modes

Now push inputs at it that a human intake nurse would handle instantly. Your job in the cell
below is to add two more probes of your own that expose a *different* kind of missing
knowledge than the ones already listed. Run it first, read the table, then add yours.

In [ ]:
PROBES = [
    ("My chest hurts",
     "the ordinary case"),
    ("My chest does not hurt",
     "negation: the opposite complaint, nearly the same reply"),
    ("I stopped taking my warfarin three days ago and I am coughing up blood",
     "an emergency, handled with the same shrug as everything else"),
    ("asdf my qwerty hurts",
     "no concept of anatomy: a nonsense word is treated like a body part"),
    ("I can't stop the bleeding",
     "an emergency turned into a therapy question by a rule that fits it syntactically"),

    # TODO: add two probes of your own. Ideas: an internal contradiction across two
    #       turns, a units error, a symptom that only matters in combination with
    #       another one, a patient who says something in the third person.
]

probe_rng = np.random.default_rng(5200)
for utterance, what_it_misses in PROBES:
    print("INPUT :", utterance)
    print("ELIZA :", respond(utterance, probe_rng))
    print("MISSES:", what_it_misses)
    print("-" * 78)

The negation pair is the one to remember. *"My chest hurts"* and *"My chest does not hurt"*
are clinically opposite and produce structurally identical output, because the program has no
representation of the *state of the world* that the sentence describes — only of the string.
Everything in this course from Week 2 onward is an attempt to build that missing layer: a
representation you can draw inferences over, so that asserting P and asserting not-P lead
somewhere different.

### The Turing Test question

ELIZA passed, for a while, with people who knew they were talking to a program. Weizenbaum's
secretary asked him to leave the room. If a system's observable behavior is what we test, and
this is what produces that behavior, then either the test is measuring something other than
understanding, or understanding is cheaper than we thought. Hold that tension — it is the
argument the whole semester is organized around, and we will be back at it in Week 13 when
the system in the box is an LLM instead of 20 lines of regex.

## Baseline: what understanding would require

Fill in the string below with your own answer, in a few sentences. There is no grading and no
right answer today. We will open this same question again in the Week 13 notebook, after
you have built a model checker, an ontology, a rule engine, and a Bayesian network — and you
will get to see whether a semester of doing this changed your mind.

In [ ]:
understanding_requirements = """
TODO: replace this text.

For a system to genuinely "understand" a patient saying "my chest hurts when I climb
stairs", I think it would have to be able to:

1.
2.
3.

And here is a test I would accept as evidence that it does:
"""

print(understanding_requirements)
print("Save this cell's text somewhere you will still have it in December.")

## Discussion

1. ELIZA has no memory across turns. Suppose we gave it perfect memory of the whole
   conversation and nothing else — no world model, no clinical knowledge. Which of the
   failures in Part 4 would that fix, and which would survive?
2. A vendor demonstrates an intake chatbot to your health system. What single question would
   you ask, or what single input would you type into the demo, that would most quickly tell
   you whether there is anything underneath it? Would the vendor let you type it?
3. The catch-all rule is what makes ELIZA never fall silent. Modern chat systems also never
   fall silent. Is fluent output in the absence of knowledge a bug, a UI decision, or a
   safety problem — and does the answer change when the user is a patient rather than a
   clinician?

## Solutions

Completed versions of the Part 2 and Part 4 TODOs. These are markdown, not runnable cells,
so scrolling ahead does not overwrite what you wrote.

**Part 2 — the two missing rules**, inserted immediately before the catch-all:

```python
    (r".*\bi (?:take|am taking|'m taking) (.*)",
     ["How long have you been taking {0}?", "Who prescribed {0}?"]),

    (r".*\b(?:mother|father|mom|dad|family|sister|brother)\b",
     ["Tell me more about your family."]),
```

Two things to notice. The medication rule captures everything after "take", so
`"I take 400 units of insulin before every meal"` returns the dose verbatim — the rule has no
idea which part of the fragment is the drug. The family rule captures nothing at all: it has
no group, so `match.groups()` is empty and `template.format()` gets no arguments. A rule that
ignores its input entirely is still, by the Turing Test's standard, a conversational move.

**Part 4 — two more probes:**

```python
    ("I am fine, my wife made me come in",
     "third person: the complaint belongs to someone not in the conversation"),
    ("I told you already, the pain is in my jaw",
     "no memory: 'already' refers to a turn ELIZA cannot access"),
```